# Exceptions

In [4]:
import sys
from pathlib import Path

current = Path.cwd()
for parent in [current, *current.parents]:
    if (parent / '_config.yml').exists():
        project_root = parent  # ← Add project root, not chapters
        break
else:
    project_root = Path.cwd().parent.parent

sys.path.insert(0, str(project_root))

from shared import thinkpython, diagram, jupyturtle
from shared.download import download

# Register as top-level modules so direct imports work in subsequent cells
sys.modules['thinkpython'] = thinkpython
sys.modules['diagram'] = diagram
sys.modules['jupyturtle'] = jupyturtle

**Learning goals:** By the end of this chapter you will be able to:

- Distinguish between syntax errors, runtime errors, and semantic errors
- Read and interpret Python tracebacks to locate the source of a bug
- Handle exceptions gracefully with `try`, `except`, `else`, and `finally`
- Catch specific exception types and provide a meaningful response to each
- Raise exceptions with `raise` and define custom exception classes
- Apply print-statement debugging and `assert` to isolate bugs systematically
- Use the `logging` module to record diagnostic messages at appropriate severity levels

## Exception Handling

When writing programs, errors are inevitable. Python distinguishes between 
- **syntax errors** (code that violates Python grammar rules) and 
- **exceptions** (errors detected during execution). 
Exception handling allows your program to respond gracefully to runtime errors instead of crashing.

**Common Exception Types**

Python has many built-in exception types. Here are the most common:

| Exception | Description | Example |
|-----------|-------------|---------|
| `ValueError` | Invalid value for operation | `int("hello")` |
| `TypeError` | Wrong type for operation | `"5" + 5` |
| `ZeroDivisionError` | Division by zero | `10 / 0` |
| `IndexError` | Index out of range | `[1, 2][5]` |
| `KeyError` | Dictionary key not found | `{}["missing"]` |
| `FileNotFoundError` | File doesn't exist | `open("nonexistent.txt")` |
| `AttributeError` | Attribute doesn't exist | `"text".nonexistent()` |
| `NameError` | Variable not defined | `print(undefined_var)` |


When executing a program and exceptions happen, without **exception handling**:
- Programs **crash** with cryptic error messages
- **Users** have poor experience
- Difficult to **debug** production issues

With exception handling, we can:
- Programs can recover from errors
- Provide user-friendly error messages
- Log errors for debugging


### `try`-`except`

The `try`-`except` statement lets you catch and respond to exceptions instead of letting them crash your program. You wrap the risky code in a `try` block; if an exception is raised, Python jumps immediately to the matching `except` block and executes it instead.

**Syntax:**
The syntax of `try`-`except` is:
```python
try:
    # Code that might raise an exception
    risky_code()
except ExceptionType:
    # Handle the exception
    handle_error()
```

A complete `try` statement can include up to four clauses, each serving a distinct role:

- **`try`**: Block containing code that might raise an exception
- **`except`**: Block to handle specific exception type(s)
- **`else`** (optional): Block that executes if no exception occurs
- **`finally`** (optional): Block that always executes, regardless of exception

In [5]:
%%expect ZeroDivisionError
# Example 1: Simple try-except

try:
    result = 10 / 0
except ZeroDivisionError:
    print("Error: Cannot divide by zero!") ### program will not crash, error is handled
else:
    print(f"Result: {result}")

Error: Cannot divide by zero!


In this example, dividing by zero raises a `ZeroDivisionError`. The `except` clause catches it and prints a friendly message, so the program continues instead of crashing. Because an exception was raised, the `else` clause is skipped.

In [ ]:
# Example 2: Practical example - safe file handling
def read_file_safely(filename):
    content = None
    try:
        with open(filename, "r") as file:
            content = file.read()
            print(f"File '{filename}' content:\n{content}")
    except FileNotFoundError:    ### may occur if the file does not exist
        print(f"Error: File '{filename}' not found!")
    except IOError as e:         ### may occur if there is an I/O error while reading the file
        print(f"Error reading file: {e}")
    else:
        print("File reading completed successfully.")
    finally:
        print("File operation finished.\n")
    return content

## Debugging

### Understanding Errors

When you write programs, things will go wrong. Learning to find and fix these problems is called **debugging**. Before you can handle errors gracefully, you need to understand what types of errors exist and how to find them.

**Types of Errors:**

| Error Type | When It Occurs | Example |
|------------|----------------|---------|
| **Syntax Error** | Code violates Python grammar | Missing colon, unmatched quotes |
| **Runtime Error (Exceptions)** | Error during execution | Division by zero, file not found |
| **Logic (Semantic) Error** | Code runs but produces wrong results | Incorrect algorithm, wrong formula |

When Python encounters a runtime error, it raises an **exception**. Without proper handling, exceptions cause programs to crash with error messages called **tracebacks**. See the exception type reference table at the start of this chapter.

### Reading Tracebacks

When Python encounters a runtime error, it displays a **traceback** - an error report that shows where the error occurred. Learning to read tracebacks is essential for debugging.

The error message includes a **traceback**, which shows:
1. **Where** the error occurred  
2. Which **lines** were executed leading to the error
3. What **type of error** happened

The order of functions in the traceback matches the order of function calls: you read it from **bottom to top**.

- **Bottom**: The error type and message (e.g., "NameError: name 'cat' is not defined")
- **Middle**: The line where the error actually happened  
- **Top**: The chain of function calls that led there

In [ ]:
# Example: Demonstrating traceback reading and debugging

def print_twice(value):
    """Print a value twice - contains a deliberate bug for demonstration."""
    print(value)
    print(cat)          # Bug: 'cat' is not defined

def cat_twice():
    """Function that calls print_twice."""
    line1 = "Bing tiddle "
    line2 = "tiddle bang."
    print_twice(line1)
    print_twice(line2)


In [8]:
# Uncomment the line below to see the traceback
# cat_twice()

print("The above would produce a traceback like this:")
print("""
Traceback (most recent call last):
  File "debug_example.py", line 15, in <module>
    cat_twice()
  File "debug_example.py", line 11, in cat_twice
    print_twice(line1)
  File "debug_example.py", line 4, in print_twice
    print(cat)
NameError: name 'cat' is not defined
""")



The above would produce a traceback like this:

Traceback (most recent call last):
  File "debug_example.py", line 15, in <module>
    cat_twice()
  File "debug_example.py", line 11, in cat_twice
    print_twice(line1)
  File "debug_example.py", line 4, in print_twice
    print(cat)
NameError: name 'cat' is not defined



In [9]:
# Debugging steps:
print("Debugging steps:")
print("1. Read from bottom up: NameError on line 4 in print_twice()")
print("2. The undefined variable is 'cat'") 
print("3. Trace back: cat_twice() called print_twice() with line1")
print("4. Fix: Change 'cat' to 'value' in print_twice()")

print("\n" + "="*50 + "\n")


Debugging steps:
1. Read from bottom up: NameError on line 4 in print_twice()
2. The undefined variable is 'cat'
3. Trace back: cat_twice() called print_twice() with line1
4. Fix: Change 'cat' to 'value' in print_twice()




### Debugging Best Practices

Debugging can be frustrating, but it is also challenging, interesting, and sometimes even fun. It is one of the most important skills you can learn.

**Debugging is like detective work**: You are given clues, and you have to infer the events that led to the results you see.

**Debugging is like experimental science**: Once you have an idea about what is going wrong, you modify your program and try again. If your hypothesis was correct, you can predict the result of the modification. If your hypothesis was wrong, you have to come up with a new one.

**Key debugging principles:**
- Start with a working program and make small modifications
- Test frequently - don't write too much code before testing
- Use print statements to trace execution
- Read error messages carefully
- Think systematically about what could be wrong

If you find yourself spending a lot of time debugging, that's often a sign you're writing too much code before testing. Taking smaller steps can help you move more quickly.

### Debugging Techniques

In [10]:
def debug_with_print_statements():
    """Demonstrate print statement debugging."""
    print("=== Print Statement Debugging ===")
    
    def calculate_average(numbers):
        print(f"DEBUG: Input numbers = {numbers}")
        print(f"DEBUG: Length = {len(numbers)}")
        
        total = 0
        for i, num in enumerate(numbers):
            total += num
            print(f"DEBUG: Step {i+1}: added {num}, total now = {total}")
        
        average = total / len(numbers)
        print(f"DEBUG: Final average = {average}")
        return average
    
    # Test with example data
    test_numbers = [10, 20, 30, 40]
    result = calculate_average(test_numbers)
    print(f"Result: {result}")

debug_with_print_statements()

print("\n" + "="*50 + "\n")



=== Print Statement Debugging ===
DEBUG: Input numbers = [10, 20, 30, 40]
DEBUG: Length = 4
DEBUG: Step 1: added 10, total now = 10
DEBUG: Step 2: added 20, total now = 30
DEBUG: Step 3: added 30, total now = 60
DEBUG: Step 4: added 40, total now = 100
DEBUG: Final average = 25.0
Result: 25.0




In [11]:
def debug_with_assertions():
    """Demonstrate assertion debugging."""
    print("=== Assertion Debugging ===")
    
    def factorial(n):
        # Assertions help catch problems early
        assert isinstance(n, int), f"Expected int, got {type(n)}"
        assert n >= 0, f"Expected non-negative number, got {n}"
        
        print(f"Computing factorial of {n}")
        
        if n == 0 or n == 1:
            return 1
        
        result = 1
        for i in range(2, n + 1):
            result *= i
            # Assertion to check intermediate results
            assert result > 0, f"Unexpected negative result at step {i}"
        
        return result
    
    # Test cases
    test_cases = [5, 0, 1]
    for test in test_cases:
        try:
            result = factorial(test)
            print(f"factorial({test}) = {result}")
        except AssertionError as e:
            print(f"Assertion failed for factorial({test}): {e}")

debug_with_assertions()



=== Assertion Debugging ===
Computing factorial of 5
factorial(5) = 120
Computing factorial of 0
factorial(0) = 1
Computing factorial of 1
factorial(1) = 1


In [12]:
print("\n" + "="*50 + "\n")

def systematic_debugging_approach():
    """Demonstrate systematic debugging methodology."""
    print("=== Systematic Debugging Approach ===")
    
    # Bug-prone function for demonstration
    def find_max_in_nested_lists(nested_list):
        """Find maximum value in nested lists - has bugs to debug."""
        print("Step 1: Check input")
        print(f"Input: {nested_list}")
        
        max_val = None
        for i, sublist in enumerate(nested_list):
            print(f"Step 2.{i+1}: Processing sublist {i}: {sublist}")
            
            for j, val in enumerate(sublist):
                print(f"  Step 2.{i+1}.{j+1}: Checking value {val}")
                
                if max_val is None:
                    max_val = val
                    print(f"  First value found: {max_val}")
                elif val > max_val:
                    max_val = val
                    print(f"  New max found: {max_val}")
        
        print(f"Step 3: Final result: {max_val}")
        return max_val
    
    # Test with various cases
    test_cases = [
        [[1, 3, 2], [7, 1, 9], [4, 5]],
        [[], [2, 8], [6]],
        [[]]
    ]
    
    for i, test_case in enumerate(test_cases):
        print(f"\n--- Test Case {i+1} ---")
        try:
            result = find_max_in_nested_lists(test_case)
            print(f"Maximum value: {result}")
        except Exception as e:
            print(f"Error occurred: {e}")
            print("Debug: This error helps us find edge cases!")

systematic_debugging_approach()



=== Systematic Debugging Approach ===

--- Test Case 1 ---
Step 1: Check input
Input: [[1, 3, 2], [7, 1, 9], [4, 5]]
Step 2.1: Processing sublist 0: [1, 3, 2]
  Step 2.1.1: Checking value 1
  First value found: 1
  Step 2.1.2: Checking value 3
  New max found: 3
  Step 2.1.3: Checking value 2
Step 2.2: Processing sublist 1: [7, 1, 9]
  Step 2.2.1: Checking value 7
  New max found: 7
  Step 2.2.2: Checking value 1
  Step 2.2.3: Checking value 9
  New max found: 9
Step 2.3: Processing sublist 2: [4, 5]
  Step 2.3.1: Checking value 4
  Step 2.3.2: Checking value 5
Step 3: Final result: 9
Maximum value: 9

--- Test Case 2 ---
Step 1: Check input
Input: [[], [2, 8], [6]]
Step 2.1: Processing sublist 0: []
Step 2.2: Processing sublist 1: [2, 8]
  Step 2.2.1: Checking value 2
  First value found: 2
  Step 2.2.2: Checking value 8
  New max found: 8
Step 2.3: Processing sublist 2: [6]
  Step 2.3.1: Checking value 6
Step 3: Final result: 8
Maximum value: 8

--- Test Case 3 ---
Step 1: Check in

## Multiple Exception Handling

You can handle multiple exception types in a single try/except block using several approaches:

1. **Multiple except blocks**: Handle each exception type differently
2. **Single except with tuple**: Handle multiple types the same way  
3. **Generic except**: Catch any exception (use cautiously)

In [13]:
# Example 4: Multiple exception handling approaches

def calculate_from_strings(num1_str, num2_str, operation):
    """Perform calculations with comprehensive error handling."""
    
    try:
        # Convert strings to numbers
        num1 = float(num1_str)
        num2 = float(num2_str)
        
        # Perform the requested operation
        if operation == 'add':
            result = num1 + num2
        elif operation == 'subtract':
            result = num1 - num2
        elif operation == 'multiply':
            result = num1 * num2
        elif operation == 'divide':
            result = num1 / num2
        else:
            raise ValueError(f"Unknown operation: {operation}")
            
        return result
        
    except ValueError as e:
        print(f"ValueError: {e}")
        return None
    except ZeroDivisionError:
        print("ZeroDivisionError: Division by zero is not allowed")
        return None
    except Exception as e:
        print(f"Unexpected error: {type(e).__name__}: {e}")
        return None

# Test different scenarios
test_cases = [
    ("10", "5", "add"),       # Valid
    ("10", "abc", "add"),     # ValueError (invalid number)
    ("10", "0", "divide"),    # ZeroDivisionError
    ("10", "5", "power"),     # ValueError (unknown operation)
]

print("Testing calculator with error handling:")
for num1, num2, op in test_cases:
    result = calculate_from_strings(num1, num2, op)
    if result is not None:
        print(f"{num1} {op} {num2} = {result}")
    print("-" * 40)

Testing calculator with error handling:
10 add 5 = 15.0
----------------------------------------
ValueError: could not convert string to float: 'abc'
----------------------------------------
ZeroDivisionError: Division by zero is not allowed
----------------------------------------
ValueError: Unknown operation: power
----------------------------------------


## Else and Finally Blocks

The try/except statement can include `else` and `finally` blocks for additional control:

- **`else` block**: Runs only if NO exception occurred in the try block
- **`finally` block**: ALWAYS runs, whether an exception occurred or not

**Complete Syntax:**
```python
try:
    # Code that might raise an exception
    risky_code()
except SpecificException:
    # Handle specific exception
    handle_error()
else:
    # Code that runs only if no exception occurred
    success_code()
finally:
    # Code that always runs (cleanup)
    cleanup_code()
```

In [14]:
# Example 5: Using else and finally blocks

def file_processor(filename):
    """Demonstrate else and finally blocks."""
    print(f"\nProcessing file: {filename}")
    
    try:
        # Simulate file processing
        if filename == "missing.txt":
            raise FileNotFoundError("File does not exist")
        elif filename == "corrupted.txt":
            raise ValueError("File is corrupted")
        else:
            print(f"Successfully opened {filename}")
            data = f"Contents of {filename}"
            
    except FileNotFoundError as e:
        print(f"File error: {e}")
        data = None
    except ValueError as e:
        print(f"Data error: {e}")
        data = None
    else:
        # This runs only if no exception occurred
        print("File processed successfully!")
        print(f"Data length: {len(data)}")
    finally:
        # This always runs
        print("Cleaning up resources...")
        print("File processor finished")
    
    return data

# Test different scenarios
test_files = ["document.txt", "missing.txt", "corrupted.txt"]

for filename in test_files:
    result = file_processor(filename)
    print(f"Returned: {result}")
    print("=" * 50)


Processing file: document.txt
Successfully opened document.txt
File processed successfully!
Data length: 24
Cleaning up resources...
File processor finished
Returned: Contents of document.txt

Processing file: missing.txt
File error: File does not exist
Cleaning up resources...
File processor finished
Returned: None

Processing file: corrupted.txt
Data error: File is corrupted
Cleaning up resources...
File processor finished
Returned: None


In [15]:
### Exercise: Safe Division with Cleanup
#   Write `safe_divide(a, b)` that:
#   1. Returns the result of `a / b`.
#   2. If `b` is zero, prints "Error: cannot divide by zero" and returns None.
#   3. Always prints "Operation complete." in a `finally` block, regardless of outcome.
### Your code starts here.




### Your code ends here.

In [16]:
### Solution

def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Error: cannot divide by zero")
        result = None
    finally:
        print("Operation complete.")
    return result

print(safe_divide(10, 2))   # 5.0  (then "Operation complete.")
print(safe_divide(5, 0))    # error message, then "Operation complete.", then None

Operation complete.
5.0
Error: cannot divide by zero
Operation complete.
None


## Raising Custom Exceptions

Sometimes you need to raise your own exceptions when certain conditions are met. Use the `raise` statement to create exceptions.

**Syntax:**
```python
raise ExceptionType("Error message")
```

**Common scenarios for raising exceptions:**
- Input validation fails
- Business logic violations
- Resource constraints
- Invalid states

In [17]:
# Example 6: Raising custom exceptions

def validate_age(age):
    """Validate age input with custom exceptions."""
    if not isinstance(age, (int, float)):
        raise TypeError(f"Age must be a number, got {type(age).__name__}")
    
    if age < 0:
        raise ValueError("Age cannot be negative")
    
    if age > 150:
        raise ValueError("Age cannot be greater than 150")
    
    print(f"Valid age: {age}")
    return age

# Test age validation
test_ages = [25, -5, 200, "twenty", 45.5, None]

for age in test_ages:
    try:
        validate_age(age)
    except (TypeError, ValueError) as e:
        print(f"Validation failed for {age}: {e}")

print("\n" + "="*50)

# Example 7: Custom exception classes
class InsufficientFundsError(Exception):
    """Custom exception for banking operations."""
    def __init__(self, balance, amount):
        self.balance = balance
        self.amount = amount
        super().__init__(f"Insufficient funds: balance={balance}, requested={amount}")

class BankAccount:
    """Simple bank account with exception handling."""
    
    def __init__(self, initial_balance=0):
        if initial_balance < 0:
            raise ValueError("Initial balance cannot be negative")
        self.balance = initial_balance
    
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.balance += amount
        print(f"Deposited ${amount}. New balance: ${self.balance}")
    
    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise InsufficientFundsError(self.balance, amount)
        
        self.balance -= amount
        print(f"Withdrew ${amount}. New balance: ${self.balance}")
    
    def get_balance(self):
        return self.balance

# Test the bank account
print("Testing bank account:")
try:
    account = BankAccount(100)
    account.deposit(50)
    account.withdraw(30)
    account.withdraw(200)  # This will raise an exception
    
except ValueError as e:
    print(f"Input error: {e}")
except InsufficientFundsError as e:
    print(f"Banking error: {e}")
    print(f"Available balance: ${e.balance}")

Valid age: 25
Validation failed for -5: Age cannot be negative
Validation failed for 200: Age cannot be greater than 150
Validation failed for twenty: Age must be a number, got str
Valid age: 45.5
Validation failed for None: Age must be a number, got NoneType

Testing bank account:
Deposited $50. New balance: $150
Withdrew $30. New balance: $120
Banking error: Insufficient funds: balance=120, requested=200
Available balance: $120


In [18]:
### Exercise: Custom Exception Class
#   1. Define `NegativeValueError(ValueError)`. Its `__init__(self, value)` should
#      store `self.value` and pass a descriptive message to `super().__init__`.
#   2. Write `square_root(x)` that raises `NegativeValueError` if x < 0,
#      otherwise returns x ** 0.5.
#   3. Test: square_root(9) → 3.0; square_root(-4) → raises NegativeValueError.
### Your code starts here.




### Your code ends here.

In [19]:
### Solution
class NegativeValueError(ValueError):
    def __init__(self, value):
        self.value = value
        super().__init__(f"Cannot take square root of negative number: {value}")

def square_root(x):
    if x < 0:
        raise NegativeValueError(x)
    return x ** 0.5

try:
    print(square_root(9))    # 3.0
    print(square_root(-4))   # raises NegativeValueError
except NegativeValueError as e:
    print(f"Caught: {e}, value was {e.value}")

3.0
Caught: Cannot take square root of negative number: -4, value was -4


## Logging

The `logging` module provides a flexible framework for recording diagnostic
messages during program execution. Unlike `print()`, logging lets you control
*which* messages appear, *where* they go, and *how much detail* to include —
without changing your code.

### Log Levels

| Level | Value | When to use |
|---|---|---|
| `DEBUG` | 10 | Detailed diagnostic info (development only) |
| `INFO` | 20 | Confirmation that things are working as expected |
| `WARNING` | 30 | Something unexpected; program still running |
| `ERROR` | 40 | A more serious problem; something failed |
| `CRITICAL` | 50 | A severe error; program may not continue |

The default level is `WARNING` — only `WARNING`, `ERROR`, and `CRITICAL`
messages appear unless you configure a lower threshold.


In [20]:
import logging

# Show ALL levels (DEBUG and above)
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')

logging.debug('Reading config file')        # detailed diagnostics
logging.info('Server started on port 8080') # normal operations
logging.warning('Disk space is low')        # something unexpected
logging.error('Failed to connect to database')  # something failed
logging.critical('System is shutting down') # severe error


DEBUG: Reading config file
INFO: Server started on port 8080
ERROR: Failed to connect to database
CRITICAL: System is shutting down


### Logging vs `print()`

| | `print()` | `logging` |
|---|---|---|
| Severity levels | No | Yes |
| Easy to silence | No (must delete/comment) | Yes (set level higher) |
| Timestamps | No | Yes (with format string) |
| Write to file | No | Yes |
| Production-ready | No | Yes |

Best practice: use `print()` for user-facing output and `logging` for
diagnostic messages during development and production monitoring.


In [21]:
import logging

logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')

def divide(a, b):
    logging.debug(f'divide called with a={a}, b={b}')
    if b == 0:
        logging.error('Division by zero attempted')
        raise ValueError('Cannot divide by zero')
    result = a / b
    logging.info(f'Result: {result}')
    return result

print(divide(10, 2))   # 5.0


DEBUG: divide called with a=10, b=2
INFO: Result: 5.0


5.0


In [22]:
### Exercise: Add Logging
#   Rewrite `safe_sqrt(x)` to use `logging` instead of `print`:
#     - DEBUG: log the input value before computing
#     - WARNING: log a warning if x is negative
#     - ERROR: raise ValueError if x is negative
#     - INFO: log the result before returning
#   Test: safe_sqrt(25) → 5.0; safe_sqrt(0) → 0.0; safe_sqrt(-4) → ValueError.
import math
### Your code starts here.



### Your code ends here.


*Try it yourself before looking at the solution below.*

---

*Try it yourself before looking at the solution below.*

---

*Try it yourself before looking at the solution below.*

---

In [23]:
# Example 2: Practical example — safe file handling
def read_file_safely(filename):
    content = None
    try:
        with open(filename, "r") as file:
            content = file.read()
            print(f"File '{filename}' content:\n{content}")
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found!")
    except IOError as e:
        print(f"Error reading file: {e}")
    else:
        print("File reading completed successfully.")
    finally:
        print("File operation finished.\n")
    return content

# Demonstrate the function
read_file_safely(str(project_root / "data" / "text_file.txt"))  # file exists
read_file_safely("no_such_file.txt")                             # does not exist

DEBUG: safe_sqrt(25)
INFO: Result: 5.0
DEBUG: safe_sqrt(0)
INFO: Result: 0.0
DEBUG: safe_sqrt(-4)


5.0
0.0
Caught: Cannot take sqrt of -4
